# DICOM Edge Inference Benchmark — Kaggle GPU

**Research-only reproducible experiment.** Independently benchmarks highdicom PR [#520](https://github.com/ImagingDataCommons/highdicom/pull/520) decoding strategies A/B/C followed by the **same untrained PyTorch CNN** on CPU and CUDA. Synthetic DICOM only — no patient datasets or clinical claims.

**Before running:**

1. In Kaggle **Settings → Accelerator**, select **GPU T4 x2**. Kaggle retired the P100 accelerator in September 2026.
2. Enable **Internet** for pip / GitHub package installation.
3. Upload `DICOM_Edge_Inference_Benchmark_Kaggle.zip` as a **private Kaggle Dataset**, then attach it using **Add Input**. If Kaggle has already extracted it, that is supported too.
4. Run cells in order. **Pilot** runs first by default. Use `PROFILE = "exp002"` only after the instrumentation smoke test and local tests succeed.

**Scope:** The experiment uses **one CUDA device (GPU 0)** for inference. It is **not** multi-GPU benchmarking and **not** a physical Jetson measurement. Highdicom decoding remains on CPU.

In [ ]:
PROFILE = "pilot"  # pilot, full, validation2, exp002_smoke, or exp002
RUN_PACKAGE_TESTS = True
# All inputs are synthetic; the package generates them in /kaggle/working/.
PACKAGE_ARCHIVE_NAME = "DICOM_Edge_Inference_Benchmark_Kaggle.zip"
HIGH_DICOM_PIN = "d42f664006a9cd2194eda0bf5b1cf0fa2fb0188e"


## 1. Verify GPU availability before installing anything

Kaggle T4 x2 allocates two GPUs, but this benchmark deliberately uses `cuda:0` only to keep its single-GPU comparison controlled.

In [ ]:
import sys, subprocess, json, os
import torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torch CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA accelerator. In Kaggle Settings select GPU T4 x2 and restart the notebook.")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
torch.cuda.set_device(0)
subprocess.run(["nvidia-smi", "-L"], check=False)


## 2. Locate and extract the attached source package

This notebook does **not** assume that a GitHub repository has already been created. Attach the packaged ZIP to Kaggle first.

In [ ]:
from pathlib import Path
import shutil, zipfile
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
PROJECT = WORK_ROOT / 'DICOM-Edge-Inference-Benchmark'

if not (PROJECT / 'pyproject.toml').exists():
    archives = sorted(INPUT_ROOT.rglob(PACKAGE_ARCHIVE_NAME))
    if archives:
        print('Extracting:', archives[0])
        with zipfile.ZipFile(archives[0]) as zip_file:
            zip_file.extractall(WORK_ROOT)
    else:
        candidates = [p for p in INPUT_ROOT.rglob('pyproject.toml') if p.parent.name == PROJECT.name]
        if candidates:
            print('Copying extracted dataset:', candidates[0].parent)
            shutil.copytree(candidates[0].parent, PROJECT, dirs_exist_ok=True)
        else:
            raise FileNotFoundError(
                'Source package not attached. Upload DICOM_Edge_Inference_Benchmark_Kaggle.zip '
                'as a private Kaggle Dataset, then Add Input to this notebook.'
            )
assert (PROJECT / 'pyproject.toml').exists(), f'Project missing at {PROJECT}'
print('Project ready:', PROJECT)


## 3. Install the benchmark and the exact highdicom contribution commit

Keep Kaggle's existing CUDA-enabled PyTorch installation. Do **not** upgrade or replace torch for this benchmark. Network access is needed to install the pinned fork revision.

In [ ]:
import subprocess
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{PROJECT}[monitoring,test]'],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     f'git+https://github.com/swapins/highdicom.git@{HIGH_DICOM_PIN}'],
    check=True,
)
import highdicom
from highdicom.io import ImageFileReader
assert callable(getattr(ImageFileReader, 'read_frames', None)), 'Expected highdicom selective decoding API'
print('highdicom version:', highdicom.__version__)
print('highdicom code:', highdicom.__file__)
# Inspect installed VCS provenance; this is not a replacement for test evidence.
import importlib.metadata
provenance = importlib.metadata.distribution('highdicom').read_text('direct_url.json')
print('Installation provenance:', provenance)
if provenance is None or HIGH_DICOM_PIN not in provenance:
    raise RuntimeError('Unable to verify the pinned highdicom commit in installed package metadata')
assert torch.cuda.is_available(), 'CUDA became unavailable after installation'


## 4. Local tests

If tests fail, do not publish speedup claims. The benchmark runner also compares SHA-256 hashes of decoded frames across all three modes.

In [ ]:
if RUN_PACKAGE_TESTS:
    subprocess.run(
        [sys.executable, '-m', 'pytest', '-q', str(PROJECT / 'tests')],
        cwd=PROJECT,
        check=True,
    )


## 5. Run A / B / C on native and JPEG DICOM — CPU and CUDA

- **A** — existing per-frame reader + inference
- **B** — selective multi-frame decoding + inference
- **C** — selective decoding with bounded prefetch + inference

Pilot: 2 encodings x 2 devices x 1 selection, with 1 warmup and 2 repeats. Full: 2 encodings x 2 devices x 5 selections, with 3 warmups and 10 repeats. Validation Run 2: 2 encodings x 2 devices x 4 batch sizes (4, 8, 16, 32), using `uniform:32`, with 3 warmups and 10 repeats per condition. Results saved to `/kaggle/working/dicom_edge_results`. No real patient data is involved.

In [ ]:
RESULTS = WORK_ROOT / 'dicom_edge_results'
subprocess.run(
    [sys.executable, str(PROJECT / 'scripts' / 'run_kaggle.py'),
     '--profile', PROFILE, '--out', str(RESULTS),
     '--frames', '96', '--rows', '128', '--columns', '128',
     '--batch-size', '16', '--prefetch', '2'],
    cwd=PROJECT,
    check=True,
)


## 6. Review results

Comparisons are **within the same encoding, device, and selection**, never across different workloads. A value above 1.0 for `speedup_vs_A` indicates a faster median than A.

In [ ]:
import pandas as pd
results = pd.read_csv(RESULTS / 'summary.csv')
display(results[['encoding', 'device', 'selection', 'mode', 'median_total_ms', 'median_fps', 'speedup_vs_A']])
with open(RESULTS / 'machine.json', encoding='utf8') as f:
    print(json.dumps(json.load(f)['available_gpus'], indent=2))


In [ ]:
import matplotlib.pyplot as plt
# Each chart is independent and plotted with Matplotlib defaults.
for device in ('cpu', 'cuda'):
    subset = results.loc[(results.device == device) & (results.encoding == 'jpeg')]
    fig, ax = plt.subplots(figsize=(8, 4))
    for mode in ('A', 'B', 'C'):
        group = subset.loc[subset['mode'] == mode].copy()
        if len(group):
            ax.plot(group['frames_selected'], group['median_total_ms'], marker='o', label=mode)
    ax.set_title(f'JPEG end-to-end latency on Kaggle {device.upper()}')
    ax.set_xlabel('Selected frames')
    ax.set_ylabel('Median end-to-end latency (ms)')
    ax.legend(title='Strategy')
    ax.grid(alpha=0.3)
    plt.show()


## 7. Export synthetic results

Download the output ZIP from Kaggle's **Output** pane. Keep the benchmark repository separate from highdicom PR #520. The notebook reports Kaggle cloud-GPU performance; it does **not** establish Jetson / Raspberry Pi performance, energy per inference, or clinical diagnostic validity.

In [ ]:
archive = shutil.make_archive(str(WORK_ROOT / 'DICOM_Edge_Kaggle_Results'), 'zip', root_dir=RESULTS)
print('Results archive:', archive)
